### 검색 성능 최적화를 위한 벡터 DB 구축

In [2]:
%pip install -Uqqq langchain langchain-openai langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### Pinecone Index 생성

In [ ]:
from pinecone import Pinecone, ServerlessSpec  # Pinecone 클라이언트, 인덱스 스펙설정

pc = Pinecone()
print(pc.list_indexes().names())  # 인덱스 이름

# 인덱스명 ir
if 'ir' not in pc.list_indexes().names():
    # 없으면 인덱스 생성
    pc.create_index(  
        name = 'ir',
        dimension=1536,
        metric='cosine',  # 유사도 기준
        spec=ServerlessSpec(
            region='us-east-1',
            cloud='aws'
        )
    )
    print('ir 인덱스 생성 완료')

else:
    print('ir 인덱스 이미 존재')

[]
ir 인덱스 생성 완료


### VectorStore 연결

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')  # 1536차원 임베딩 모델

# 벡터스토어 연결
vector_store = PineconeVectorStore(
    index_name = 'ir',  # 연결할 index명
    embedding = embeddings  # 사용할 임베딩 함수 (연결할 인덱스 차원과 임베딩 차원이 같아야 함)
)

### Document Upsert

In [7]:
import pandas as pd 

df = pd.read_csv('documents.csv')
df

,idx,doc_id,title,content
0,0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(..."
1,1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기..."
2,2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet..."
3,3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...
4,4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...
5,5,D6,2024년 기후 변화 종합 보고서,"2024년 전 지구 평균 기온은 산업화 이전 대비 약 1.2℃ 상승했으며, 해수면 ..."
6,6,D7,AI 기술 동향 및 윤리,"최근 인공지능 분야에서는 생성형 AI, 멀티모달 모델, 강화학습 기반 에이전트 개발..."
7,7,D8,서울 지하철 이용 가이드,"서울 지하철은 1호선부터 9호선까지 운행되며, 주요 환승역으로는 서울역·강남역·종로..."
8,8,D9,판소리 “춘향가” 서사 구조,"판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작품에는 “춘향가..."
9,9,D10,한국 축구 대표팀 주요 기록,"한국 축구 대표팀은 2002 한일 월드컵 4강 진출, 2012 런던 올림픽 동메달 ..."


In [ ]:
# DataFrame -> Langchain Document 변환
from langchain_core.documents import Document  # Langchain 문서 객체

docs_to_index = []
for idx, row in df.iterrows():  # df을 행 단위로 순회
    doc_id = row['doc_id']  # 문서 ID 추출
    content = row['content']  # 문서 본문 추출
    doc = Document(page_content=content, metadata={'doc_id': doc_id})  # Document 형식 저장
    docs_to_index.append(doc)  # 리스트에 추가

print(len(docs_to_index))

30


In [9]:
# 벡터스토어에 문서 업서트 : 임베딩 후 저장 (기존에 있으면 업데이트 Update, 없으면 인설트 Insert)
vector_store.add_documents(docs_to_index)

['ee4512c9-adf8-4054-98e8-33c6e48ed19e',
 'a1981104-5638-4f3b-8730-f27364abb608',
 '48568f4b-0567-47dc-bbf3-59581b4e4f73',
 '0801d856-45aa-4fba-b3d4-ff8015e8efb5',
 'fba018ea-a93b-4e2f-be01-687dd46909f4',
 '6ba4d88a-adac-4c7b-8071-a95ea50d4000',
 'bce9538c-5fa4-4ac9-8da7-f86511b79297',
 '52a147e9-a9a2-4bd6-803f-f940b80fbe77',
 '64ee23ef-20d6-4dd6-b550-699f17112d60',
 '7b4c354b-a4cc-4530-8da1-777e0aa03062',
 'cee0fd4e-12a6-4def-ac9f-f9583fe887dd',
 '9f03682c-c4ba-4612-9578-f555e54495bc',
 '1c3c9ae6-b4e0-41a1-91f0-455978f44c11',
 '225b0a31-42b5-481a-a91e-155e9fab761b',
 '5e0af08c-8879-4bec-ac47-48ac4b6079ac',
 '14ecc60d-eadb-4d09-b6d9-322223cb33c7',
 'fe7fd2e4-8cb1-4c01-811a-496090ff2208',
 '275b6b85-9ad7-4ea5-b8d0-1fc7cc9e95b3',
 'ccd99d22-f3b3-4364-958f-f93c5c771ad5',
 'c0a6c6db-cab1-46c7-867d-a6097e8d8ee1',
 'b281f333-4b5e-40b6-90f2-3db017d7938b',
 '97541dea-6d86-493d-b18e-8623da08e764',
 '8e251c16-a5f3-4906-a405-7a236466b0c8',
 '45891cab-8fac-4172-892d-4de06b0a5cef',
 'bccfc1fb-9a4c-

In [ ]:
query = '제주도 여행'

results = vector_store.similarity_search_with_score(query, k=5)  # [(유사 문서1, 점수), ...]
for rank, (doc, score) in enumerate(results, 1):
    print(f'{rank} Score: {score}')
    print(f'doc_id: {doc.metadata['doc_id']}')
    print(f'content: {doc.page_content}')
    print()

1 Score: 0.507941484
doc_id: D1
content: 제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...

2 Score: 0.439489365
doc_id: D12
content: 서울 근교에서 당일치기로 다녀올 만한 여행지로는 가평 쁘띠프랑스, 남양주 수종사, ...

3 Score: 0.281069219
doc_id: D13
content: 비빔밥은 지역별로 칼로리, 탄수화물, 단백질, 지방 함량이 차이를 보입니다. 전주 ...

4 Score: 0.235480785
doc_id: D17
content: 2023년 한국 영화 흥행 순위 Top10에는 “헌트”, “비상선언”, “범죄도시3...

5 Score: 0.20054765
doc_id: D9
content: 판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작품에는 “춘향가...

